In [1]:
"""
Lomb-Scargle Period Analysis for FITS Light Curves
----------------------------------------------------
- Reads all .fits files recursively from root_dir using lightkurve
- Computes Lomb-Scargle periodogram
- Detects top 5 prominent peaks (period + power)
- Checks for double-dipper signature
- Saves results to a CSV file
"""

import re
import numpy as np
import pandas as pd
import lightkurve as lk
from pathlib import Path
from astropy.timeseries import LombScargle
from scipy.signal import find_peaks


root_dir   = Path("/Users/suyuxin/downloads/MiaTESS")
output_dir = Path("/Users/suyuxin/Downloads/light_curvesMia")
output_dir.mkdir(exist_ok=True)

OUTPUT_CSV = output_dir / "ls_results.csv"

# Periodogram settings
MIN_PERIOD       = 0.1    # days
MAX_PERIOD       = 20.0   # days
SAMPLES_PER_PEAK = 10     # frequency grid oversampling

# Peak-finding settings
PERIOD_EPSILON = 0.1      # fractional tolerance for duplicate periods
MIN_HEIGHT     = 0.0      # minimum LS power to be considered a peak
N_PEAKS        = 5        # number of top peaks to record

# Double-dipper thresholds
DD_PERIOD_TOL  = 0.15     # how close P2/P1 must be to 2.0 (fractional)
DD_POWER_RATIO = 0.5      # power2 must be >= this fraction of power1

MIN_DATAPOINTS = 100      # skip files with fewer valid points


def extract_tic_and_sector(fits_path):
    """
    Parses e.g. 'tess2020186164531-s0027-0000000299798795-0189-a_fast-lc.fits'
    → tic_id = '299798795', sector = '27'
    """
    name = Path(fits_path).name
    sector_match = re.search(r"-s(\d+)-", name)
    tic_match    = re.search(r"-s\d+-(\d+)-", name)
    sector = str(int(sector_match.group(1))) if sector_match else "unknown"
    tic_id = str(int(tic_match.group(1)))    if tic_match    else "unknown"
    return tic_id, sector


def find_top_peaks(periods, power,
                   max_period=MAX_PERIOD,
                   period_epsilon=PERIOD_EPSILON,
                   min_height=MIN_HEIGHT,
                   n_peaks=N_PEAKS):
    periods = np.asarray(periods)
    power   = np.asarray(power)

    m = (periods < max_period) & (periods > 0) & np.isfinite(periods) & np.isfinite(power)
    if not np.any(m):
        return np.full(n_peaks, np.nan), np.full(n_peaks, np.nan)

    p  = periods[m]
    pw = power[m]

    peaks, _ = find_peaks(pw, height=min_height)
    if peaks.size == 0:
        return np.full(n_peaks, np.nan), np.full(n_peaks, np.nan)

    # sort candidate peaks by descending power
    order = peaks[np.argsort(pw[peaks])[::-1]]

    kept = []
    for idx in order:
        per = p[idx]
        if all(np.abs(per - p[k]) > period_epsilon * per for k in kept):
            kept.append(idx)
        if len(kept) == n_peaks:
            break

    out_p  = np.full(n_peaks, np.nan)
    out_pw = np.full(n_peaks, np.nan)
    out_p [:len(kept)] = p [kept]
    out_pw[:len(kept)] = pw[kept]
    return out_p, out_pw


def check_double_dipper(peak_periods, peak_powers,
                        dd_period_tol=DD_PERIOD_TOL,
                        dd_power_ratio=DD_POWER_RATIO):
    """
    If P2 ≈ 2 × P1 (within dd_period_tol) AND power2 >= dd_power_ratio × power1
    → classified as double dipper, true period = P2.
    Otherwise → not a double dipper, true period = P1.
    """
    p1,  p2  = peak_periods[0], peak_periods[1]
    pw1, pw2 = peak_powers[0],  peak_powers[1]

    if np.isnan(p1) or np.isnan(p2):
        return False, p1

    period_match = abs(p2 / p1 - 2.0) < dd_period_tol * 2.0
    power_ok     = pw2 >= dd_power_ratio * pw1

    if period_match and power_ok:
        return True, p2
    return False, p1


fits_files = sorted(root_dir.rglob("*.fits"))
print(f"Found {len(fits_files)} FITS files\n")

if len(fits_files) == 0:
    raise RuntimeError("No FITS files found — check root_dir path!")

results = []

for fpath in fits_files:
    print(f"  Processing: {fpath.name}")
    tic_id, sector = extract_tic_and_sector(fpath)

    # Default values in case something fails
    peak_periods = np.full(N_PEAKS, np.nan)
    peak_powers  = np.full(N_PEAKS, np.nan)
    is_dd        = False
    true_period  = np.nan

    try:
        lc   = lk.read(str(fpath))
        time = lc.time.value
        flux = lc.flux.value

        mask = np.isfinite(time) & np.isfinite(flux)
        time = time[mask]
        flux = flux[mask]

        print(f"    Valid points: {len(time)}")

        if len(time) >= MIN_DATAPOINTS:

            flux = flux / np.nanmedian(flux) - 1.0

            ls = LombScargle(time, flux)
            frequency, power = ls.autopower(
                minimum_frequency=1.0 / MAX_PERIOD,
                maximum_frequency=1.0 / MIN_PERIOD,
                samples_per_peak=SAMPLES_PER_PEAK
            )
            periods = 1.0 / frequency

            peak_periods, peak_powers = find_top_peaks(periods, power)
            is_dd, true_period = check_double_dipper(peak_periods, peak_powers)

            print(f"    Best period: {peak_periods[0]}")

        else:
            print("    Too few datapoints — still writing row with NaNs")

    except Exception as e:
        print(f"    ERROR: {e}")
        print("    Writing row with NaNs")

    # Always append a row
    row = {
        "filename": fpath.name,
        "tic_id": tic_id,
        "sector": sector,
        "is_double_dipper": is_dd,
        "true_period_d": true_period
    }

    for i in range(N_PEAKS):
        row[f"period_peak{i+1}_d"] = peak_periods[i]
        row[f"power_peak{i+1}"]    = peak_powers[i]

    results.append(row)

print(f"\nRows collected: {len(results)}")

'''df = pd.DataFrame(results)

if df.empty:
    raise RuntimeError("DataFrame is empty — something is wrong earlier!")

df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved to {OUTPUT_CSV}")


fits_files = sorted(root_dir.rglob("*.fits"))
print(f"Found {len(fits_files)} FITS files\n")

results = []

for fpath in fits_files:
    print(f"  Processing: {fpath.name}")
    tic_id, sector = extract_tic_and_sector(fpath)

    try:
        # ── Load with lightkurve (handles all TESS column naming automatically)
        lc   = lk.read(str(fpath))
        time = lc.time.value
        flux = lc.flux.value

        # Remove NaNs
        mask = np.isfinite(time) & np.isfinite(flux)
        time = time[mask]
        flux = flux[mask]

        if len(time) < MIN_DATAPOINTS:
            print(f"    Skipping: too few data points ({len(time)})")
            continue

        # Normalise flux (divide by median so it hovers around 1.0,
        # then subtract 1 to centre on zero for the periodogram)
        flux = flux / np.nanmedian(flux) - 1.0

        print(f"    TIC ID: {tic_id} | Sector: {sector} | "
              f"Points: {len(time)} | "
              f"Time: {time.min():.2f}–{time.max():.2f} d")

        # ── Lomb-Scargle periodogram
        ls = LombScargle(time, flux)
        frequency, power = ls.autopower(
            minimum_frequency=1.0 / MAX_PERIOD,
            maximum_frequency=1.0 / MIN_PERIOD,
            samples_per_peak=SAMPLES_PER_PEAK
        )
        periods = 1.0 / frequency

        # Find top peaks
        peak_periods, peak_powers = find_top_peaks(periods, power)

        # Double-dipper check
        is_dd, true_period = check_double_dipper(peak_periods, peak_powers)

        print(f"    Best period: {peak_periods[0]:.4f} d  "
              f"| Double dipper: {is_dd}"
              + (f"  → true period: {true_period:.4f} d" if is_dd else ""))

    except Exception as e:
        print(f"    WARNING: could not process {fpath.name}: {e}")
        peak_periods = np.full(N_PEAKS, np.nan)
        peak_powers  = np.full(N_PEAKS, np.nan)
        is_dd, true_period = None, np.nan

    # ── Build result row
    row = {"filename": fpath.name, "tic_id": tic_id, "sector": sector}
    for i in range(N_PEAKS):
        row[f"period_peak{i+1}_d"] = peak_periods[i]
        row[f"power_peak{i+1}"]    = peak_powers[i]
    row["is_double_dipper"] = is_dd
    row["true_period_d"]    = true_period
    results.append(row)'''

# ── Save CSV
df = pd.DataFrame(results)
df.to_csv(OUTPUT_CSV, index=False, float_format="%.6f")

print(f"Double dippers found: {df['is_double_dipper'].sum()} / {len(df)}")
print(df[["filename", "tic_id", "sector", "period_peak1_d",
          "power_peak1", "is_double_dipper", "true_period_d"]].to_string(index=False))

Found 10 FITS files

  Processing: tess2020186164531-s0027-0000000299798795-0189-a_fast-lc.fits
    Valid points: 100736
    Best period: 1.22939991841265
  Processing: tess2020186164531-s0027-0000000299799658-0189-a_fast-lc.fits
    Valid points: 100736
    Best period: 8.963414247024717
  Processing: tess2020212050318-s0028-0000000299798795-0190-a_fast-lc.fits
    Valid points: 89372
    Best period: 1.2269561547887604
  Processing: tess2020212050318-s0028-0000000299799658-0190-a_fast-lc.fits
    Valid points: 91105
    Best period: 3.2592950328797747
  Processing: tess2020294194027-s0031-0000000004646810-0198-s_lc.fits
    Valid points: 16170
    Best period: 11.710559591952865
  Processing: tess2021146024351-s0039-0000000299798795-0210-a_fast-lc.fits
    Valid points: 116263
    Best period: 0.6142902099680206
  Processing: tess2023181235917-s0067-0000000299798795-0261-a_fast-lc.fits
    Valid points: 85556
    Best period: 1.2288951417332974
  Processing: tess2023209231226-s0068-0